# Introduction
In machine learning, <i>bias</i> refers to the production of consistently prejudiced outcomes by the algorithm, which can create unfair circumstances on multiple fronts. In this blog post, we aim to explore that <i>bias</i> by:
<ol>
    <li> Creating a machine learning model that predicts an individual characteristic like: employment status or income, on the basis of other demographic characteristics
    <li> Performing a "fairness audit" in order to assess whether our algorithm displays bias with respect to demographic characteristics (race or sex)
</ol>

For the purpose of this blog post, we will be using the <u>folktables</u> package to download data on which we will perform the <i>bias audit</i>. Here, we will be using the data of individuals from the state of Alabama. 

# Loading and Setting Up the Data
First, let us load the data and prepare it for our purposes:

In [1]:
from folktables import ACSDataSource, ACSEmployment, BasicProblem, adult_filter
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split 
from sklearn.tree import DecisionTreeClassifier

STATE = 'AL'

data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')

acs_data = data_source.get_data(states=[STATE], download = True)

Now, let us examine the first few rows of the data, to get an idea of it:

In [2]:
acs_data.head()

,RT,SERIALNO,DIVISION,SPORDER,PUMA,REGION,ST,ADJINC,PWGTP,AGEP,...,PWGTP71,PWGTP72,PWGTP73,PWGTP74,PWGTP75,PWGTP76,PWGTP77,PWGTP78,PWGTP79,PWGTP80
0,P,2018GQ0000049,6,1,1600,3,1,1013097,75,19,...,140,74,73,7,76,75,80,74,7,72
1,P,2018GQ0000058,6,1,1900,3,1,1013097,75,18,...,76,78,7,76,80,78,7,147,150,75
2,P,2018GQ0000219,6,1,2000,3,1,1013097,118,53,...,117,121,123,205,208,218,120,19,123,18
3,P,2018GQ0000246,6,1,2400,3,1,1013097,43,28,...,43,76,79,77,80,44,46,82,81,8
4,P,2018GQ0000251,6,1,2701,3,1,1013097,16,25,...,4,2,29,17,15,28,17,30,15,1


Each row in the above dataset corresponds to an individual citizen of Alabama who filled out the PUMS survey of 2018. Therefore, what we have is a $n$x$p$ matrix, where $n$ = number of data points, and $p$ = number of features. We can see that there are a bunch of features in this dataset, and all of them might not be pertinent for our analysis. Therefore, we are only going to choose the features which are relevant for this blog post: 

In [3]:
possible_features=['AGEP', 'SCHL', 'MAR', 'RELP', 'DIS', 'ESP', 'CIT', 'MIG', 'MIL', 'ANC', 'NATIVITY', 'DEAR', 'DEYE', 'DREM', 'SEX', 'RAC1P', 'ESR']

Now, since we want to predict <i>ESR</i> - Employment Status, based on every relevant feature except the <i>RAC1P</i> - Race, we want to remove these two features from the list of <i>possible_features</i>. This is because one of these features is our target (output), and the we are excluding the other one (race) to study racial bias in our machine learning model: 

In [4]:
features_to_use = [f for f in possible_features if f not in ["ESR", "RAC1P"]]

Now that we know:
<ol>
    <li> Features (features_to_use): the features we wish to use for the prediction
    <li> Target (ESR): the thing we are trying to predict
    <li> Group (RAC1P): the group by which we wish to audit the bias
</ol>

We can go ahead and create a <i>BasicProblem</i>:

In [5]:
EmploymentProblem = BasicProblem(
    features=features_to_use,
    target='ESR',
    target_transform=lambda x: x == 1,
    group='RAC1P',
    preprocess=lambda x: x,
    postprocess=lambda x: np.nan_to_num(x, -1),
)

Now we can go ahead and extract our feature matrix, output labels vector, and the group (race) vector as numpy objects:

In [6]:
features, label, group = EmploymentProblem.df_to_numpy(acs_data)

Now, that we have all the necessary data, we can split our data into <i>training data</i> and <i>testing data</i>. Here, 80% of the data will be used for training and the remaining 20% will be used for testing:

In [7]:
X_train, X_test, y_train, y_test, group_train, group_test = train_test_split(features, label, group, test_size=0.2, random_state=0)

# Basic Descriptives
Now, that we have our data set up, we can address some basic descriptives of the data which will aid us in our analysis later on. For this it is useful to make a <i>data frame</i> out of our data:

In [8]:
df = pd.DataFrame(X_train, columns = features_to_use)
df["group"] = group_train
df["label"] = y_train

Question 1: How many individuals are in the data?

In [9]:
print(f"Answer 1: There are {df.shape[0]} people in the training data set!")

Answer 1: There are 38221 people in the training data set!


Question 2: Of these individuals, what proportion have target label equal to 1? In employment prediction, these would correspond to employed individuals.

In [10]:
employed = df[df['label']==True]
print(f"Answer 2: There are {employed.shape[0]} number of people in the training data set who are employed - have target label equal to True!")

Answer 2: There are 15638 number of people in the training data set who are employed - have target label equal to True!


Question 3: Of these individuals, how many are in each of the groups?
<br><br>Answer 3: We can see a breakdown of how many people of each racial group are employed:

In [11]:
print(employed.groupby('group').size())

group
1    12020
2     2976
3       68
4        1
5        6
6      226
7        3
8      147
9      191
dtype: int64


Question 4: In each group, what proportion of individuals have target label equal to 1?
<br><br>Answer 4: The proportion of individuals in each group who are employed (have target label 1):

In [12]:
print(df.groupby(['group'])[['label']].aggregate([np.mean]).round(2))

      label
       mean
group      
1      0.42
2      0.37
3      0.44
4      1.00
5      0.21
6      0.50
7      0.27
8      0.37
9      0.29


Question 5: Check for intersectional trends by studying the proportion of positive target labels broken out by your chosen group labels and an additional group label. For example, if you chose race (RAC1P) as your group, then you could also choose sex (SEX) and compute the proportion of positive labels by both race and sex. 
<br><br> Answer 5: For answering this question, first let us look at the positive labels broken down only by SEX. Then we can look at the intersectionality of RACE and SEX, by looking at positive labels broken down by these two group labels.

In [13]:
print(df.groupby(['SEX'])[['label']].aggregate([np.mean]).round(2))

    label
     mean
SEX      
1.0  0.45
2.0  0.37


We can see that: 45% of MEN (1.0) are employed, while 37% of WOMEN (2.0) are employed! Now let us look at the positive target labels broken down by both RACE and SEX:

In [14]:
print(df.groupby(['group','SEX'])[['label']].aggregate([np.mean]).round(2))

          label
           mean
group SEX      
1     1.0  0.47
      2.0  0.38
2     1.0  0.36
      2.0  0.38
3     1.0  0.48
      2.0  0.39
4     1.0  1.00
5     1.0  0.23
      2.0  0.19
6     1.0  0.62
      2.0  0.41
7     1.0  0.50
      2.0  0.14
8     1.0  0.46
      2.0  0.28
9     1.0  0.32
      2.0  0.25


We can visualize this data using a bar chart for easier understanding: 

In [15]:
print("Hello")

Hello


# Training the Model

# Auditing the Model 

# Concluding Discussions